# 09.7 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/07-how-llms-work/solutions.ipynb)

> **About the outputs in this notebook.** They were produced with `llama3.2` (3B) via Ollama on a laptop. Yours will differ: a different model, a different run, even the same model on a different machine. Small local models make more mistakes than large hosted ones; that is part of what this lesson shows.

## Exercise 1

After *the*: mug 3, lamp 2 → P(*mug* | *the*) = 3/5. After *is*: red 2, big 2, blue 1 → P(*red* | *is*) = 2/5. Greedy decoding writes *the mug is …* and then has a **tie** between *red* and *big*; `most_common` breaks it by first appearance, so it writes *the mug is red*.

In [1]:
from collections import Counter, defaultdict

sentences = ["the mug is red", "the mug is big", "the lamp is red",
             "the lamp is big", "the mug is blue"]
counts = defaultdict(Counter)                 # counts[word][next_word]
for s in sentences:
    words = s.split()
    for a, b in zip(words, words[1:]):
        counts[a][b] += 1

for word, nxt in counts.items():
    total = sum(nxt.values())
    print(f"after '{word}':", {w: f"{c}/{total}" for w, c in nxt.items()})

after 'the': {'mug': '3/5', 'lamp': '2/5'}
after 'mug': {'is': '3/3'}
after 'is': {'red': '2/5', 'big': '2/5', 'blue': '1/5'}
after 'lamp': {'is': '2/2'}


In [2]:
word, text = "the", ["the"]
while word in counts:
    word = counts[word].most_common(1)[0][0]
    text.append(word)
print(" ".join(text))

the mug is red


## Exercise 2

In [3]:
import numpy as np

candidates = ["order", "purchase", "patience", "banana"]
scores = np.array([3.0, 2.5, 1.5, -1.0])       # the model's raw scores (logits)

def softmax(z, temperature=1.0):
    e = np.exp(z / temperature)
    return e / e.sum()

for T in [0.5, 1.0, 2.0]:
    probs = softmax(scores, T)
    print(f"T={T}: " + "  ".join(f"{w} {p:.3f}" for w, p in zip(candidates, probs)))

T=0.5: order 0.705  purchase 0.259  patience 0.035  banana 0.000
T=1.0: order 0.541  purchase 0.328  patience 0.121  banana 0.010
T=2.0: order 0.419  purchase 0.326  patience 0.198  banana 0.057


In [4]:
for T in [0.1, 5.0]:
    print(f"T={T}: " + "  ".join(f"{w} {p:.3f}" for w, p in zip(candidates, softmax(scores, T))))

T=0.1: order 0.993  purchase 0.007  patience 0.000  banana 0.000
T=5.0: order 0.323  purchase 0.292  patience 0.239  banana 0.145


At T = 0.1 the scores become 30, 25, 15, −10: *order* takes 99.3% and the rest almost nothing. As T → 0 softmax turns into "always pick the top token" (greedy). At T = 5 the scores become 0.6, 0.5, 0.3, −0.2 and the probabilities move towards equal (0.32, 0.29, 0.24, 0.15). As T → ∞ every candidate becomes equally likely, which produces nonsense.

## Exercise 3

In [5]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
samples = {"English": "Hand-painted ceramic mug with a blue wave pattern, 350 ml.",
           "French": "Tasse en céramique peinte à la main avec un motif de vague bleue, 350 ml.",
           "order number": "4921 0387 5512 6604",
           "email": "jordan.okafor+orders@shopwave.example"}
for name, s in samples.items():
    n = len(enc.encode(s))
    print(f"{name:12} {len(s):3} chars {n:3} tokens  {len(s)/n:.1f} chars per token")

English       58 chars  15 tokens  3.9 chars per token
French        73 chars  23 tokens  3.2 chars per token
order number  19 chars  11 tokens  1.7 chars per token
email         37 chars  11 tokens  3.4 chars per token


English packs the most characters into each token (3.9). The French translation needs 23 tokens for what English says in 15. The order number is by far the most expensive per character (1.7): long digit strings are split into small groups. The email address sits in between.

## Exercise 4

In [6]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [7]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

LLM provider: ollama · model: llama3.2


In [8]:
prompt = ("Is this review positive or negative? Reply with one word. "
          "Review: The lamp arrived broken.")
for T in [0.0, 1.2]:
    answers = [ask(prompt, temperature=T, max_tokens=10).strip().strip(".").lower() for _ in range(5)]
    print(f"temperature {T}: {answers}")

temperature 0.0: ['negative', 'negative', 'negative', 'negative', 'negative']


temperature 1.2: ['negative', 'negative', 'negative', 'negative', 'negative']


When one answer is overwhelmingly likely, even a high temperature rarely changes it: sampling still picks the dominant token almost every time. Temperature matters where the model has several plausible continuations (names, wording, ideas), not where the answer is obvious. For classification you still set it to 0, so that the rare odd answer cannot happen.